In [8]:
import sys
import torch
import torchaudio
from pathlib import Path
import numpy as np

from asteroid.metrics import get_metrics
# Add project root to Python path
project_root = Path.cwd()
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'hstasnet'))

from hstasnet.hstasnet import HSTasNet
from src.states import  load_model_from_package
import time

class StemEvaluator:
    def __init__(self, model_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.model_package = torch.load(model_path, map_location=device)
        self.model = load_model_from_package(self.model_package)
        self.model = self.model.to(device)
        self.model.eval()
        
    def load_and_validate_audio(self, audio_path, segment_length):
        """Load and validate audio format"""
        audio, sr = torchaudio.load(audio_path)
        
        # Handle channels
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)
        elif audio.dim() > 2:
            audio = audio[:2]
            
        # Handle length
        if audio.shape[1] < segment_length:
            pad_length = segment_length - audio.shape[1]
            audio = torch.nn.functional.pad(audio, (0, pad_length))
            
        # Move to device
        audio = audio.to(self.device)
        return audio, sr
    

    def calculate_metrics(self, mixture,  result):
        """Calculate SDR for a single track result"""
        metrics = {}
        # Get predictions and ground truth
        pred_sources = np.array([
            result['predictions'][stem].squeeze().numpy()
            for stem in ['drums', 'bass', 'other', 'vocals']
        ])
        true_sources = np.array([
            result['ground_truth'][stem].squeeze().numpy()
            for stem in ['drums', 'bass', 'other', 'vocals']
        ])

        # Ensure both sources have the same shape
        min_length = min(pred_sources.shape[-1], true_sources.shape[-1])
        pred_sources = np.array([ps[..., :min_length] for ps in pred_sources])
        true_sources = np.array([ts[..., :min_length] for ts in true_sources])

        # collapse mixture to mono
        mixture = mixture.mean(dim=1)

        # collapse sources to mono
        pred_sources = pred_sources.mean(axis=1)
        true_sources = true_sources.mean(axis=1)
        # pad the mixture to the same length as the sources
        mixture = torch.nn.functional.pad(mixture, (0, true_sources.shape[-1] - mixture.shape[-1]))

        mixture = mixture.cpu().numpy()
        # Print shapes for debugging
        print("Pred sources shape:", pred_sources.shape)
        print("True sources shape:", true_sources.shape)
        print("Mixture shape:", mixture.shape)
        # asteoid's get_metrics expects sources in shape (nsrc, nsamples), we have stereo arrays in the form of (n_instruments, 2, n_samples)
        # so we need to drop the channel dimension and collapse the sources to mono e.g
        # true_sources.shape = (n_instruments, n_samples) 
        # pred_sources.shape = (n_instruments, n_samples)
        # mixture.shape = (1, n_samples)
    
        # Calculate SDR
        metrics_dict = get_metrics(mix=mixture, clean=true_sources, estimate=pred_sources, sample_rate=result['sample_rate'], metrics_list=[ 'sdr', 'sir', 'sar'])

        # Store metrics
        metrics = {
            'file': result['track_name'],
            **metrics_dict
        }
        return metrics

    def process_track(self, track_path, stem_names= ['bass', 'drums', 'other', 'vocals'], segment_length_in_s=20.0):
        track_path = Path(track_path)
        sr = 44100
        segment_length = int(sr * segment_length_in_s)
        fade_length = int(sr * 1.0)
        perf_stats = {'segments': [], 'summary': {}}
        mixture, sr = self.load_and_validate_audio(track_path / "mixture.wav", segment_length)
        start_time = time.perf_counter()
        
        # Time preprocessing
        prep_start = time.perf_counter()
        fade_transform = torchaudio.transforms.Fade(
            fade_in_len=fade_length,
            fade_out_len=fade_length,
            fade_shape='linear'
        )
        mixture = fade_transform(mixture).to(self.device)
        mixture = mixture.unsqueeze(0)
        prep_time = time.perf_counter() - prep_start
        
        # Time inference
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        inference_start = time.perf_counter()
        # do naive inference: pass our whole mixture through the model
        with torch.no_grad():
            predictions = self.model(mixture)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        inference_time = time.perf_counter() - inference_start
        
        # Record performance stats
        perf_stats['segments'].append({
            'prep_time': prep_time,
            'inference_time': inference_time,
            'total_time': time.perf_counter() - start_time,
            'gpu_memory': torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
        })
        
        # Calculate summary
        perf_stats['summary'] = {
            'avg_inference_time': inference_time,
            'realtime_factor': inference_time / segment_length_in_s
            
        }
        
        # Add performance stats to existing results structure
        results = {
            "track_name": track_path.name,
            "predictions": {
                stem: predictions[0, i].detach().cpu()
                for i, stem in enumerate(stem_names)
            },
            "sample_rate": sr,
            "ground_truth": {},
            "perf_stats": perf_stats
        }
        
        # Load ground truth stems
        for stem in stem_names:
            audio, _ = self.load_and_validate_audio(track_path / f"{stem}.wav", segment_length)
            results["ground_truth"][stem] = fade_transform(audio).cpu()
            
        # Calculate metrics
        metrics = self.calculate_metrics(mixture, results)
        # add the metrics to the results
        results = {**results, **metrics}
        return results
        

# Resolve symlink for the musdb18 dataset
ds_path = Path('~/.datasets/musdb18hq').expanduser().resolve(strict=True)

# Define the train and test paths
train_path = ds_path / 'train'
test_path = ds_path / 'test'

# Dictionary defining the stem file names
stems = {
    "drums": "drums.wav",
    "bass": "bass.wav",
    "other": "other.wav",
    "vocals": "vocals.wav",
    "mixture": "mixture.wav"
}

# Print paths to ensure correctness
print(f"Dataset Path: {ds_path}")
print(f"Train Path: {train_path}")
print(f"Test Path: {test_path}")

# Grab the first 5 samples from the test set
num_samples = 1
test_samples = sorted(test_path.glob('*'))[:num_samples]

samples = {
    sample.stem: {stem: sample / stems[stem] for stem in stems}
    for sample in test_samples
}
# Usage
models_dir = Path("./out/models/")
model_path = next(models_dir.glob("*.pt"))
evaluator = StemEvaluator(model_path, device="cpu")

# Process test samples
results = []
# restrict to one sample
for sample in samples:
    print(f"Processing sample {samples[sample]}")
    track_path = samples[sample]["mixture"]
    print(f"Processing track {track_path}")
    results.append(evaluator.process_track(str(track_path.parent), segment_length_in_s=10.0))


Dataset Path: /media/stepincto/musdb18hq
Train Path: /media/stepincto/musdb18hq/train
Test Path: /media/stepincto/musdb18hq/test
Processing sample {'drums': PosixPath('/media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/drums.wav'), 'bass': PosixPath('/media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/bass.wav'), 'other': PosixPath('/media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/other.wav'), 'vocals': PosixPath('/media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/vocals.wav'), 'mixture': PosixPath('/media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/mixture.wav')}
Processing track /media/stepincto/musdb18hq/test/AM Contra - Heart Peripheral/mixture.wav


/tmp/ipykernel_717381/3856209299.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model_package = torch.load(model_path, map_location=device)


Pred sources shape: (4, 9265664)
True sources shape: (4, 9265664)
Mixture shape: (1, 9265664)


In [9]:
import os
from pathlib import Path

def save_results(result, track, output_dir="output"):
    """Save audio results to an output directory."""
    output_path = Path(output_dir) / result['track_name']
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\n=== Saving Results for Track: {result['track_name']} ===")
    sr = result['sample_rate']
    
    # Save mixture
    mixture_path = Path(track) / "mixture.wav"
    if mixture_path.exists():
        print("Saving mixture...")
        mixture, _ = torchaudio.load(mixture_path)
        torchaudio.save(output_path / "mixture.wav", mixture, sr)
    
    # Save ground truth and predictions
    for stem in result['predictions'].keys():
        print(f"Saving {stem}...")
        
        # Ground truth
        if stem in result['ground_truth']:
            gt_audio = result['ground_truth'][stem]
            gt_file = output_path / f"{stem}_gt.wav"
            torchaudio.save(gt_file, gt_audio, sr)
        
        # Prediction
        pred_audio = result['predictions'][stem]
        pred_file = output_path / f"{stem}.wav"
        torchaudio.save(pred_file, pred_audio, sr)
    
    print(f"Results saved to {output_path}")

# Usage
for result, track in zip(results, test_samples):
    save_results(result, track)


=== Saving Results for Track: AM Contra - Heart Peripheral ===
Saving mixture...
Saving bass...
Saving drums...
Saving other...
Saving vocals...
Results saved to output/AM Contra - Heart Peripheral


In [12]:
import pandas as pd
from pathlib import Path

def save_metrics(results, output_dir="output"):
    """Save metrics to a CSV file."""

    for result in results:
        metrics = {

            "track_name": result["track_name"],
            "sample_rate": result["sample_rate"],
            "input_sdr": result["input_sdr"],
            "input_sir": result["input_sir"],
            "input_sar": result["input_sar"],
            "sdr": result["sdr"],
            "sir": result["sir"],
            "sar": result["sar"],
        }

        # Convert metrics to DataFrame
        df = pd.DataFrame([metrics])

        # Ensure output directory exists
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Save to CSV
        csv_path = output_path / "metrics.csv"
        df.to_csv(csv_path, index=False)
        print(f"Metrics saved to {csv_path}")
save_metrics(results, output_dir="output")

Metrics saved to output/metrics.csv
